# Fraus - Fine-tune do BERTimbau para classificacao de emocao

Este notebook treina o classificador das **sete classes de emocao** que o
requisito de banca pede: as seis basicas de Ekman (1992) mais o neutro.

**A setima emocao, o desprezo, NAO e treinada aqui** -- ela e derivada da diade
primaria raiva + nojo, como Plutchik (1980) a define, em
`fraus.sinais.emocao.desprezo_derivado`. Motivo: nenhum corpus rotulado em
portugues anota desprezo, e treinar so essa classe em corpus de outra
procedencia ensinaria o modelo a reconhecer o *estilo do texto* em vez da
emocao -- o mesmo vazamento que a latencia disjunta causou no primeiro fusor.

Requer **GPU**: `Ambiente de execucao > Alterar tipo de ambiente de execucao > GPU`.

Este notebook e INDEPENDENTE do 01 e do 04. So o notebook 02 depende dele,
porque e o 02 que monta o vetor de features completo.

Sequencia:

1. GPU conferida e Drive montado;
2. pacote `fraus` instalado -- a ordem das classes vem de la, nunca redigitada;
3. `go_emotions_ptbr` carregado, agregado por anotador e reduzido a Ekman;
4. treino com **pesos de classe**, porque o corpus e desequilibrado 43:1;
5. avaliacao interna, **por classe** e nao so macro;
6. avaliacao no **XED-pt**, corpus independente que nao passou por traducao automatica;
7. export para o Drive.

In [ ]:
# 1. GPU conferida e Drive montado.
!pip -q install transformers datasets accelerate evaluate scikit-learn

import torch

assert torch.cuda.is_available(), (
    'Runtime sem GPU. Ambiente de execucao > Alterar tipo de ambiente de execucao > GPU. '
    'Sao ~46 mil exemplos e tres epocas: em CPU isso nao termina.'
)

from google.colab import drive
drive.mount('/content/drive')

import os

DIR_CHECKPOINT = '/content/drive/MyDrive/fraus/checkpoints-emocao'
DIR_SAIDA = '/content/drive/MyDrive/fraus/modelos/bertimbau-emocao'
DIR_MODELOS = os.path.dirname(DIR_SAIDA)
os.makedirs(DIR_CHECKPOINT, exist_ok=True)
os.makedirs(DIR_SAIDA, exist_ok=True)
print('gpu:', torch.cuda.get_device_name(0))

## 2. O pacote `fraus`

A ordem das classes **nao e redigitada aqui**. Ela vem de
`fraus.sinais.emocao.NOMES_EMOCOES`, que e a mesma lista que o sinal usa para
ler as probabilidades do modelo. Redigitar a ordem no notebook e o jeito mais
facil de treinar `raiva` no indice 2 e ler `raiva` no indice 3 -- inverter
classe nao gera erro, so atribui a emocao errada em silencio.

O clone vai para `/content/fraus-src`, nunca `/content/fraus`: o cwd do Colab e
`/content`, e um diretorio chamado `fraus` ali vira namespace package e
sequestra o import. O caminho tambem entra em `sys.path` a mao, porque o `.pth`
do `pip install -e` so e lido quando o interpretador inicia -- e o do Colab ja
esta de pe.

In [ ]:
# 2. Pacote fraus instalado a partir do repositorio.
DONO = 'joao-ped'
NOME_REPO = 'Fraus'
# O RAMO DO TREINO. Era 'feat/motor-e-dashboard', que em 24/08/2026 estava
# 141 commits atras do main e zero a frente -- absorvido e obsoleto.
#
# O ESTRAGO era silencioso, que e o pior tipo: naquele ramo
# `montar_features` ainda aceitava UM classificador e devolvia 16 features.
# O notebook rodava sem erro, treinava feliz, e cuspia um fusor de 16
# features num repositorio cujo `NOMES_FEATURES` tem 35. O treino nao
# falhava: ele acertava o alvo errado, e so aparecia na API, no boot, como
# 'X has 35 features, but StandardScaler is expecting 16'.
#
# Se um dia precisar treinar de um ramo de trabalho, troque aqui -- mas
# confira antes que o contrato de features daquele ramo e o do destino.
RAMO = 'main'
DIR_REPO = '/content/fraus-src'

import shutil
import subprocess
import sys

# Os TRES motivos de nao haver token sao distintos, e o remedio de cada um e
# outro. `except Exception: token = None` tratava os tres como o mesmo e caia
# para um clone anonimo -- que num repo privado morre la na frente com
# "could not read Username for 'https://github.com'", mensagem que nao
# menciona Secret nenhum e manda procurar no lugar errado. O caso que mais
# acontece e o pior de diagnosticar: o segredo existe, esta correto, e o
# toggle DESTE notebook esta desligado.
token = None
motivo_sem_token = None
try:
    from google.colab import userdata
except ImportError:
    motivo_sem_token = 'nao estamos no Colab -- nao ha painel de Secrets.'
else:
    try:
        token = userdata.get('TOKEN_GITHUB')
    except Exception as erro:
        # Comparacao por NOME da classe, e nao por `except userdata.X`, de
        # proposito: essas excecoes nao sao API estavel do Colab, e um
        # AttributeError aqui derrubaria a celula por causa do TRATAMENTO do
        # erro em vez do erro.
        classe = type(erro).__name__
        if classe == 'NotebookAccessError':
            motivo_sem_token = (
                'TOKEN_GITHUB existe, mas ESTE notebook nao tem acesso a ele. '
                'Abra o painel Secrets (icone de chave, barra da esquerda) e '
                'ligue o toggle da linha.'
            )
        elif classe == 'SecretNotFoundError':
            motivo_sem_token = 'nao ha TOKEN_GITHUB no painel Secrets deste Colab.'
        else:
            motivo_sem_token = f'{classe}: {erro}'

if motivo_sem_token:
    print('AVISO -- clone SEM credencial:', motivo_sem_token)

url_publica = f'https://github.com/{DONO}/{NOME_REPO}.git'
url = f'https://{token}@github.com/{DONO}/{NOME_REPO}.git' if token else url_publica

shutil.rmtree(DIR_REPO, ignore_errors=True)
clone = subprocess.run(
    ['git', 'clone', '-q', '--branch', RAMO, url, DIR_REPO],
    capture_output=True, text=True,
)
if clone.returncode != 0:
    # A mensagem do git traz a URL: com token dentro, isso imprimiria o segredo.
    erro = clone.stderr.replace(token, '***') if token else clone.stderr
    raise RuntimeError(
        f'git clone falhou (codigo {clone.returncode}): {erro.strip()}\n'
        'Se for erro de autenticacao, o repositorio e privado: crie um token '
        'em github.com/settings/tokens com escopo `repo`, salve no painel '
        'Secrets do Colab (icone de chave) com o nome TOKEN_GITHUB e ligue o '
        'acesso para este notebook. Alternativa sem token: suba o .zip do '
        'repositorio para /content e descompacte em ' + DIR_REPO + '.'
    )

if token:
    # `git clone` GRAVA a URL com o token em .git/config. O disco do Colab e
    # efemero, mas a saida do notebook e salva no arquivo: um `!git remote -v`
    # numa celula de debug imprimiria o segredo dentro do .ipynb, que e
    # exatamente o vazamento que usar o painel de Secrets existe para evitar.
    subprocess.run(
        ['git', '-C', DIR_REPO, 'remote', 'set-url', 'origin', url_publica],
        check=True,
    )

!pip install -q -e {DIR_REPO}

if DIR_REPO not in sys.path:
    sys.path.insert(0, DIR_REPO)

commit = subprocess.check_output(
    ['git', '-C', DIR_REPO, 'rev-parse', '--short', 'HEAD']
).decode().strip()
print('fraus instalado no commit', commit)

from fraus.sinais.emocao import NOMES_EMOCOES

print(len(NOMES_EMOCOES), 'classes:', NOMES_EMOCOES)

## 3. Corpus: `go_emotions_ptbr`

Traducao PT-BR do GoEmotions (Demszky et al., 2020 -- 58 mil comentarios do
Reddit anotados em 27 emocoes + neutro), licenca Apache 2.0. O arquivo vem no
formato CRU do GoEmotions: **uma linha por ANOTADOR**, nao por exemplo. A
coluna `texto` traz a traducao; `text` e o ingles original.

### Reducao para Ekman

As 27 categorias viram 6 pelo **mapeamento oficial que o proprio GoEmotions
publica** (`ekman_mapping.json` no repositorio do Google Research), baixado
aqui em vez de copiado -- agrupamento inventado a mao seria escolha
metodologica disfarcada de detalhe tecnico.

### Agregacao dos anotadores

Um rotulo conta quando **pelo menos 2 anotadores** o marcaram. A maioria dos
exemplos tem 3 anotadores, entao 2 e a maioria simples. Medido nos dados:

| limiar | exemplos com rotulo unico |
|---|---|
| 1 anotador | 17.367 |
| **2 anotadores** | **46.014** |

O limiar mais alto rende MAIS exemplos, o que e contra-intuitivo ate ver o
motivo: com 1 anotador quase todo exemplo recebe varios rotulos Ekman
diferentes e cai fora do filtro de rotulo unico.

Exemplos que caem em mais de uma classe de Ekman sao **descartados**: a tarefa
aqui e multiclasse de rotulo unico, e forcar um vencedor entre duas emocoes
empatadas inventaria rotulo.

In [ ]:
# 3. Corpus carregado, agregado por anotador e reduzido a Ekman.
import collections
import json
import urllib.request

import pandas as pd

BASE = 'https://huggingface.co/datasets/antoniomenezes/go_emotions_ptbr/resolve/main'
URL_EKMAN = ('https://raw.githubusercontent.com/google-research/google-research/'
             'master/goemotions/data/ekman_mapping.json')

with urllib.request.urlopen(URL_EKMAN) as resposta:
    mapa_ekman = json.load(resposta)

# Os nomes em portugues sao os de NOMES_EMOCOES -- a lista do pacote manda.
INGLES_PARA_PT = {
    'joy': 'alegria', 'sadness': 'tristeza', 'anger': 'raiva',
    'fear': 'medo', 'disgust': 'nojo', 'surprise': 'surpresa',
}
fina_para_ekman = {
    fina: INGLES_PARA_PT[grosso]
    for grosso, finas in mapa_ekman.items()
    for fina in finas
}
fina_para_ekman['neutral'] = 'neutro'

faltando = set(INGLES_PARA_PT.values()) | {'neutro'}
faltando -= set(fina_para_ekman.values())
assert not faltando, f'classe sem mapeamento: {faltando}'

bruto = pd.concat(
    [pd.read_csv(f'{BASE}/goemotions_{i}_pt.csv') for i in (1, 2, 3)],
    ignore_index=True,
)
print('linhas de anotacao:', len(bruto))

# `example_very_unclear` marca o exemplo que os proprios anotadores nao
# entenderam. Treinar nele e ensinar ruido.
bruto = bruto[~bruto['example_very_unclear'].astype(bool)]

colunas_finas = [c for c in fina_para_ekman if c in bruto.columns]
assert len(colunas_finas) == 28, f'esperava 28 colunas de emocao, achei {len(colunas_finas)}'

LIMIAR_ANOTADORES = 2

votos = bruto.groupby('id')[colunas_finas].sum()
textos = bruto.groupby('id')['texto'].first()

por_ekman = collections.defaultdict(dict)
for coluna in colunas_finas:
    alvo = fina_para_ekman[coluna]
    marcados = votos.index[votos[coluna] >= LIMIAR_ANOTADORES]
    for chave in marcados:
        por_ekman[chave][alvo] = True

linhas = [
    {'texto': textos[chave], 'classe': next(iter(classes))}
    for chave, classes in por_ekman.items()
    if len(classes) == 1
]
dados = pd.DataFrame(linhas).dropna()
dados = dados[dados['texto'].str.strip().astype(bool)]

dados['rotulo'] = dados['classe'].map({nome: i for i, nome in enumerate(NOMES_EMOCOES)})
assert dados['rotulo'].notna().all(), 'classe fora de NOMES_EMOCOES'
dados['rotulo'] = dados['rotulo'].astype(int)

print('exemplos com rotulo unico:', len(dados))
print(dados['classe'].value_counts())

## 4. O desequilibrio, e por que NAO subamostrar

O corpus e desequilibrado em **43:1** -- `alegria` tem 19.284 exemplos e `nojo`
tem 444.

Os notebooks 01 e 02 balanceiam por subamostragem da classe majoritaria. Aqui
isso seria destrutivo: cortar tudo ao tamanho de `nojo` deixaria **3.108
exemplos** para sete classes, jogando fora 93% do corpus. Fine-tune de BERT com
3 mil exemplos em 7 classes nao aprende as classes raras -- so troca um
problema por outro.

**Escolha: pesos de classe na funcao de perda**, pela formula `n / (k * n_c)` --
a mesma que o `class_weight='balanced'` do scikit-learn usa. Os 46 mil exemplos
ficam, e o erro numa conversa de `nojo` custa ~43x o erro numa de `alegria`.

**A metrica que vale e o F1-MACRO, e o relatorio por classe.** A acuracia crua
aqui e enganosa: um modelo que chutasse `alegria` sempre acertaria 42% sem ter
aprendido nada.

### Consequencia para o desprezo, a declarar no relatorio

`nojo` e a classe mais rara do corpus, e o **desprezo e derivado de raiva x
nojo**. A qualidade do desprezo fica limitada de cima pela qualidade da pior
classe do modelo. Se o F1 de `nojo` sair baixo, o desprezo herda isso -- e isso
precisa aparecer no relatorio, nao ser descoberto pela banca.

In [ ]:
# 4. Tokenizacao e divisao treino/teste.
from datasets import Dataset
from transformers import AutoTokenizer

MODELO_BASE = 'neuralmind/bert-base-portuguese-cased'
TAMANHO_MAXIMO = 192  # o mesmo de fraus/sinais/emocao.py

tokenizador = AutoTokenizer.from_pretrained(MODELO_BASE)

conjunto = Dataset.from_pandas(dados[['texto', 'rotulo']].reset_index(drop=True))
conjunto = conjunto.map(
    lambda lote: tokenizador(lote['texto'], truncation=True, max_length=TAMANHO_MAXIMO),
    batched=True,
)
conjunto = conjunto.rename_column('rotulo', 'labels')

# `stratify_by_column` mantem a proporcao das classes raras no teste: sem isso,
# `nojo` poderia sumir do conjunto de avaliacao e o F1 dela viria vazio.
conjunto = conjunto.class_encode_column('labels')
divisao = conjunto.train_test_split(test_size=0.15, seed=42, stratify_by_column='labels')
print(divisao)

In [ ]:
# 5. Treino com pesos de classe.
import numpy as np
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score
from transformers import (AutoModelForSequenceClassification, DataCollatorWithPadding,
                          Trainer, TrainingArguments)

modelo = AutoModelForSequenceClassification.from_pretrained(
    MODELO_BASE, num_labels=len(NOMES_EMOCOES)
)

contagem = np.bincount(divisao['train']['labels'], minlength=len(NOMES_EMOCOES))
pesos = len(divisao['train']) / (len(NOMES_EMOCOES) * np.maximum(contagem, 1))
pesos_tensor = torch.tensor(pesos, dtype=torch.float32).to('cuda')
for nome, n, peso in zip(NOMES_EMOCOES, contagem, pesos):
    print(f'  {nome:10s} n={n:6d}  peso={peso:6.2f}')


class TreinadorPonderado(Trainer):
    """Trainer com perda ponderada por classe.

    Sem isso, o gradiente e dominado por `alegria` e o modelo aprende a nunca
    responder `nojo` -- que e exatamente a classe de que o desprezo depende.
    """

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        rotulos = inputs.pop('labels')
        saidas = model(**inputs)
        perda = nn.CrossEntropyLoss(weight=pesos_tensor)(
            saidas.logits.view(-1, len(NOMES_EMOCOES)), rotulos.view(-1)
        )
        return (perda, saidas) if return_outputs else perda


def calcular_metricas(predicao):
    preditos = np.argmax(predicao.predictions, axis=1)
    return {
        'acuracia': accuracy_score(predicao.label_ids, preditos),
        'f1_macro': f1_score(predicao.label_ids, preditos, average='macro'),
    }


argumentos = TrainingArguments(
    output_dir=DIR_CHECKPOINT,
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',  # NUNCA acuracia: com 43:1 ela premia o chute
    fp16=True,
    logging_steps=100,
)

treinador = TreinadorPonderado(
    model=modelo,
    args=argumentos,
    train_dataset=divisao['train'],
    eval_dataset=divisao['test'],
    data_collator=DataCollatorWithPadding(tokenizador),
    compute_metrics=calcular_metricas,
)
treinador.train()

In [ ]:
# 6. Avaliacao interna, POR CLASSE.
from sklearn.metrics import classification_report

predicao = treinador.predict(divisao['test'])
preditos = np.argmax(predicao.predictions, axis=1)

print(classification_report(
    predicao.label_ids, preditos,
    labels=list(range(len(NOMES_EMOCOES))),
    target_names=NOMES_EMOCOES,
    digits=4,
    zero_division=0,
))

# O F1 de `nojo` limita o desprezo, que e derivado de raiva x nojo. Este numero
# vai para o relatorio junto com a metrica do desprezo.
f1_por_classe = f1_score(predicao.label_ids, preditos, average=None,
                         labels=list(range(len(NOMES_EMOCOES))), zero_division=0)
indice_nojo = NOMES_EMOCOES.index('nojo')
indice_raiva = NOMES_EMOCOES.index('raiva')
print(f'\nteto do desprezo: F1 de nojo {f1_por_classe[indice_nojo]:.4f} '
      f'e de raiva {f1_por_classe[indice_raiva]:.4f}')

## 7. Avaliacao independente no XED-pt

O corpus de treino foi **traduzido por maquina**. Reportar so a metrica interna
mediria o quao bem o modelo aprendeu a traducao, nao a emocao em portugues.

O **XED** (Ohman et al., COLING 2020, CC-BY) anota legendas de filme com as 8
emocoes de Plutchik mais neutro. A parte em portugues tem 7.220 linhas e
**nao passou por traducao automatica** -- veio por projecao de anotacao sobre
legendas ja em portugues.

Duas ressalvas, as duas a declarar no relatorio:

1. o XED-pt e **portugues europeu** ("pa", "artola"), enquanto o treino e
   PT-BR. Diferenca de variedade, nao so de dominio;
2. `anticipation` e `trust` **nao existem em Ekman** e sao descartados; as
   linhas multirrotulo tambem, pelo mesmo motivo do treino.

Este numero vai sair **mais baixo** que o interno. Isso e esperado e e o ponto:
ele mede transferencia, e e ele que responde a pergunta "mas nao e so traducao
automatica?".

In [ ]:
# 7. XED-pt como conjunto de teste independente.
import csv
import io

URL_XED = 'https://raw.githubusercontent.com/Helsinki-NLP/XED/master/Projections/pt-projections.tsv'

# Plutchik no XED: 1 anger, 2 anticipation, 3 disgust, 4 fear, 5 joy,
# 6 sadness, 7 surprise, 8 trust. Anticipation e trust nao tem par em Ekman.
XED_PARA_PT = {
    '1': 'raiva', '3': 'nojo', '4': 'medo',
    '5': 'alegria', '6': 'tristeza', '7': 'surpresa',
}

with urllib.request.urlopen(URL_XED) as resposta:
    conteudo = resposta.read().decode('utf-8')

frases, rotulos_xed, descartadas = [], [], 0
for linha in csv.reader(io.StringIO(conteudo), delimiter='\t'):
    if len(linha) < 2:
        continue
    frase, codigos = linha[0].strip(), [c.strip() for c in linha[1].split(',')]
    classes = {XED_PARA_PT[c] for c in codigos if c in XED_PARA_PT}
    if len(classes) != 1 or not frase:
        descartadas += 1
        continue
    frases.append(frase)
    rotulos_xed.append(NOMES_EMOCOES.index(next(iter(classes))))

print(f'XED-pt: {len(frases)} frases usadas, {descartadas} descartadas '
      '(multirrotulo, ou so anticipation/trust)')
print(collections.Counter(NOMES_EMOCOES[r] for r in rotulos_xed))

xed = Dataset.from_dict({'texto': frases, 'labels': rotulos_xed}).map(
    lambda lote: tokenizador(lote['texto'], truncation=True, max_length=TAMANHO_MAXIMO),
    batched=True,
)
predicao_xed = treinador.predict(xed)
preditos_xed = np.argmax(predicao_xed.predictions, axis=1)

presentes = sorted(set(rotulos_xed))
print()
print(classification_report(
    rotulos_xed, preditos_xed,
    labels=presentes,
    target_names=[NOMES_EMOCOES[i] for i in presentes],
    digits=4,
    zero_division=0,
))
f1_xed = f1_score(rotulos_xed, preditos_xed, average='macro', labels=presentes, zero_division=0)
print(f'F1-macro no XED-pt (corpus independente): {f1_xed:.4f}')

In [ ]:
# 8. Export para o Drive.
metricas_internas = treinador.evaluate()

modelo.save_pretrained(DIR_SAIDA)
tokenizador.save_pretrained(DIR_SAIDA)

# As metricas de emocao ficam na RAIZ de modelos/, irmas do fusor.joblib e do
# metricas.json do notebook 01 -- nao dentro da pasta do modelo.
with open(f'{DIR_MODELOS}/metricas_emocao.json', 'w', encoding='utf-8') as arquivo:
    json.dump({
        'acuracia': float(metricas_internas['eval_acuracia']),
        'f1_macro': float(metricas_internas['eval_f1_macro']),
        'f1_macro_xed_pt': float(f1_xed),
        'f1_por_classe': {
            nome: float(valor) for nome, valor in zip(NOMES_EMOCOES, f1_por_classe)
        },
        'classes': NOMES_EMOCOES,
        'corpus_treino': 'go_emotions_ptbr (traducao automatica do GoEmotions)',
        'corpus_teste_independente': 'XED-pt (portugues europeu, sem traducao automatica)',
        'limiar_anotadores': LIMIAR_ANOTADORES,
        'desprezo': 'derivado da diade raiva + nojo (Plutchik 1980), nao treinado',
    }, arquivo, ensure_ascii=False, indent=2)

print('gravados:')
print(' ', DIR_SAIDA)
print(' ', f'{DIR_MODELOS}/metricas_emocao.json')
print()
print('Copie para o repo local:')
print(f'  {DIR_SAIDA}  ->  modelos/bertimbau-emocao/')
print(f'  {DIR_MODELOS}/metricas_emocao.json  ->  modelos/metricas_emocao.json')